### Bart finetune

In [1]:
#! pip install transformers datasets accelerate

preprocessing

In [8]:
! cd 

for later: bart-larger

In [15]:
# from transformers import AutoTokenizer

# tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large")

# def tokenize(example):
#     return tokenizer(example["text"], truncation=True, padding="max_length")

# dataset = dataset.map(tokenize, batched=True)
# dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

## bart-mnli preprocessing and tokenizing

In [20]:
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score, f1_score
print("loading")
# ========================
# LOAD CSV
# ========================
df = pd.read_csv(r"c:\Users\chand\Desktop\phishing-project\Phishing-detector\dataset\processed\emails_sms_combined.csv")

# sanity check
df = df[["channel", "text", "label"]]
df["label"] = df["label"].astype(int)
print(df.head())
# ========================
# CONVERT TO DATASET
# ========================
dataset = Dataset.from_pandas(df)

# ========================
# PREPROCESS (MNLI STYLE)
# ========================
def preprocess(example):
    premise = f"Channel: {example['channel']} Message: {example['text']}"
    hypothesis = "This message is phishing"

    label = 2 if example["label"] == 1 else 0  # MNLI mapping

    return {
        "premise": premise,
        "hypothesis": hypothesis,
        "label": label
    }

dataset = dataset.map(preprocess)

# ========================
# TOKENIZER
# ========================
tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-mnli")

def tokenize(example):
    return tokenizer(
        example["premise"],
        example["hypothesis"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

dataset = dataset.map(tokenize, batched=True)

# keep only required columns
dataset = dataset.remove_columns([col for col in dataset.column_names if col not in ["input_ids", "attention_mask", "label"]])
dataset.set_format(type="torch")

# ========================
# TRAIN / TEST SPLIT
# ========================
cols_to_remove = ["channel", "text", "premise", "hypothesis", "__index_level_0__"]

existing_cols = [c for c in cols_to_remove if c in dataset.column_names]

dataset = dataset.remove_columns(existing_cols)
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset["train"]
val_dataset = dataset["test"]

# ========================
# LOAD MODEL
# ========================
model = AutoModelForSequenceClassification.from_pretrained(
    "facebook/bart-large-mnli"
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# ========================
# METRICS
# ========================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)

    # convert MNLI labels → binary
    preds_bin = (preds == 2).astype(int)
    labels_bin = (labels == 2).astype(int)

    acc = accuracy_score(labels_bin, preds_bin)
    f1 = f1_score(labels_bin, preds_bin)

    return {
        "accuracy": acc,
        "f1": f1
    }


In [ ]:

# ========================
# TRAINING ARGS
# ========================
training_args = TrainingArguments(
    output_dir="./mnli-phishing",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    do_eval=True,
    logging_steps=50,
)

# ========================
# TRAINER
# ========================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

# ========================
# TRAIN
# ========================
trainer.train()

# ========================
# FINAL EVAL
# ========================
results = trainer.evaluate()
print("\nFinal Evaluation:")
print(results)

In [ ]:
trainer.save_model("./mnli-phishing-model")
tokenizer.save_pretrained("./mnli-phishing-model")

# load back

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_path = "./mnli-phishing-model"

tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

In [ ]:
import torch

def predict(email, channel="email"):
    premise = f"Channel: {channel} Message: {email}"
    hypothesis = "This message is phishing"

    inputs = tokenizer(premise, hypothesis, return_tensors="pt", truncation=True)

    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.softmax(outputs.logits, dim=-1)

    phishing_prob = probs[0][2].item()
    label = int(probs.argmax() == 2)

    return {
        "phishing_prob": phishing_prob,
        "label": label
    }